# Required Tasks Tri-Engine Notebook

本 notebook 对 `required_tasks.txt` 中的 7 个仿真任务做统一组织，并在 `qutip`、`quantumtoolbox.jl`、`quantumoptics.jl` 三个 backend 上横向运行。

结果目录固定写到 `examples/noise_simulation_tests/runs/required_tasks_tri_engine/`。

说明：
- 本 notebook 不改仓库源码，只调用现有 `run_workflow`。
- 读取线输入/输出、控制与读取链路联合影响、串扰任务，当前采用“等效参数代理”建模，而不是完整链路级电路模型。
- 2026-03-03 的本地探针显示三引擎 `trace.states` 语义尚未完全对齐，已记录为 `issues/ISSUE_DYN_P1_TRI_ENGINE_TRACE_OBSERVABLE_ALIGNMENT.md`。因此这里优先展示原始终态向量、向量长度、脉冲指标和基础观测量，避免误用现有 `observables` 字段。


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('Cannot find project root containing pyproject.toml and examples/backend.yaml')


ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.analysis.trace_semantics import state_encoding
from qsim.ui.notebook import run_workflow

ENGINES = ['qutip', 'julia_qtoolbox', 'julia_qoptics']
BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'examples' / 'noise_simulation_tests' / 'runs' / 'required_tasks_tri_engine'
SUMMARY_ROOT = OUT_ROOT / '_summaries'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)
print('ENGINES =', ENGINES)


ROOT = D:\超导量子计算机噪声抑制\qsim
BACKEND_PATH = D:\超导量子计算机噪声抑制\qsim\examples\backend.yaml
OUT_ROOT = D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine
ENGINES = ['qutip', 'julia_qtoolbox', 'julia_qoptics']


In [2]:
QASM_1Q_BASE = '''
OPENQASM 3;
include "stdgates.inc";
qubit[1] q;
bit[1] c;
h q[0];
rz(0.25) q[0];
x q[0];
measure q[0] -> c[0];
'''

QASM_1Q_READOUT = '''
OPENQASM 3;
include "stdgates.inc";
qubit[1] q;
bit[1] c;
x q[0];
measure q[0] -> c[0];
'''

QASM_2Q_BELL = '''
OPENQASM 3;
include "stdgates.inc";
qubit[2] q;
bit[2] c;
h q[0];
cx q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
'''

QASM_2Q_SPECTATOR = '''
OPENQASM 3;
include "stdgates.inc";
qubit[2] q;
bit[2] c;
x q[0];
rz(0.30) q[0];
measure q[0] -> c[0];
measure q[1] -> c[1];
'''

In [3]:
def pulse_metrics(out_dir: str | Path) -> dict[str, float]:
    out_dir = Path(out_dir)
    npz_path = out_dir / 'pulse_samples.npz'
    if not npz_path.exists():
        alts = sorted(out_dir.glob('pulse_samples*.npz'))
        if not alts:
            return {}
        npz_path = alts[0]

    data = np.load(npz_path)
    metrics: dict[str, float] = {}
    for prefix in sorted({name[:-2] for name in data.files if name.endswith('_t')}):
        t_key = f'{prefix}_t'
        y_key = f'{prefix}_y'
        if t_key not in data.files or y_key not in data.files:
            continue
        t = np.asarray(data[t_key], dtype=float)
        y = np.asarray(data[y_key], dtype=float)
        if t.size == 0 or y.size == 0:
            continue
        metrics[f'{prefix}_samples'] = float(len(t))
        metrics[f'{prefix}_duration'] = float(t[-1] - t[0]) if len(t) > 1 else 0.0
        metrics[f'{prefix}_abs_area'] = float(np.trapz(np.abs(y), t)) if len(t) > 1 else float(np.abs(y).sum())
        metrics[f'{prefix}_peak'] = float(np.max(np.abs(y)))
    return metrics


def run_single(task_tag: str, case_tag: str, engine: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, solver_mode: str = 'me') -> dict:
    out_dir = OUT_ROOT / task_tag / case_tag / engine
    result = run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        solver_mode=solver_mode,
        persist_artifacts=True,
        export_dxf=False,
        export_plots=False,
        decoder='mock',
        allow_mock_fallback=False,
    )
    return result


def summarize_result(task_tag: str, case_tag: str, engine: str, result: dict, hardware: dict, noise: dict, note: str = '') -> dict:
    trace = result['trace']
    final_state = [float(x) for x in (trace.states[-1] if trace.states else [])]
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    meta = dict(getattr(trace, 'metadata', {}) or {})
    details = dict(meta.get('details', {}) or {})
    row = {
        'task': task_tag,
        'case': case_tag,
        'engine': engine,
        'trace_engine': trace.engine,
        'state_encoding': state_encoding(trace),
        'num_qubits': int(meta.get('num_qubits', 0) or 0),
        'state_len': int(len(final_state)),
        'final_state_json': json.dumps(final_state, ensure_ascii=False),
        'final_state_sum': float(sum(final_state)) if final_state else 0.0,
        'final_state_last': float(final_state[-1]) if final_state else np.nan,
        'final_state_max': float(max(final_state)) if final_state else np.nan,
        'samples': int(len(trace.times)),
        'final_p1_obs': float(obs.get('final_p1', np.nan)),
        'final_p0_obs': float(obs.get('final_p0', np.nan)),
        'mean_excited_obs': float(obs.get('mean_excited', np.nan)),
        'solver': str(meta.get('solver', result.get('solver_mode', ''))),
        'solver_impl': str(details.get('solver_impl', '')),
        'native_solver': bool(meta.get('native_solver', False)),
        'note': note,
        'hardware_json': json.dumps(hardware, ensure_ascii=False, sort_keys=True),
        'noise_json': json.dumps(noise, ensure_ascii=False, sort_keys=True),
        'out_dir': str(result['out_dir']),
    }
    row.update(pulse_metrics(result['out_dir']))
    return row


def attach_compare_status(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    statuses = {}
    reasons = {}
    for (task, case), group in df.groupby(['task', 'case']):
        encodings = sorted(set(str(x) for x in group['state_encoding']))
        if len(encodings) == 1 and encodings[0] == 'per_qubit_excited_probability':
            statuses[(task, case)] = 'pointwise-comparable'
            reasons[(task, case)] = 'all engines expose per-qubit excited probabilities'
        else:
            statuses[(task, case)] = 'semantic-review-needed'
            reasons[(task, case)] = ' | '.join(encodings)
    df['compare_status'] = [statuses[(t, c)] for t, c in zip(df['task'], df['case'])]
    df['compare_reason'] = [reasons[(t, c)] for t, c in zip(df['task'], df['case'])]
    return df


def run_tri_cases(task_tag: str, qasm_text: str, cases: list[dict], solver_mode: str = 'me') -> pd.DataFrame:
    rows = []
    for case in cases:
        case_tag = case['tag']
        hardware = dict(case.get('hardware', {}) or {})
        noise = dict(case.get('noise', {}) or {})
        note = str(case.get('note', ''))
        for engine in ENGINES:
            result = run_single(task_tag, case_tag, engine, qasm_text, hardware=hardware, noise=noise, solver_mode=solver_mode)
            rows.append(summarize_result(task_tag, case_tag, engine, result, hardware, noise, note=note))
    df = pd.DataFrame(rows)
    df = attach_compare_status(df)
    csv_path = SUMMARY_ROOT / f'{task_tag}_summary.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f'[{task_tag}] wrote', csv_path)
    return df


def plot_metric(df: pd.DataFrame, metric: str, title: str, ylabel: str | None = None):
    usable = df[df['compare_status'] == 'pointwise-comparable']
    if usable.empty:
        print(f'{title}: skipped plot because no case is pointwise-comparable across engines.')
        return
    pivot = usable.pivot(index='case', columns='engine', values=metric)
    ax = pivot.plot(kind='bar', figsize=(9, 4), rot=0)
    ax.set_title(title)
    ax.set_ylabel(ylabel or metric)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


def display_task(df: pd.DataFrame, task_title: str, metrics: list[str]):
    cols = [
        'task', 'case', 'engine', 'state_encoding', 'compare_status', 'compare_reason',
        'state_len', 'final_state_json', 'final_state_sum', 'final_p0_obs', 'final_p1_obs', 'mean_excited_obs',
    ]
    pulse_cols = [c for c in ['RO_0_duration', 'RO_0_abs_area', 'RO_0_peak', 'XY_0_abs_area'] if c in df.columns]
    extra_cols = [c for c in ['solver_impl', 'native_solver', 'note'] if c in df.columns]
    show = [c for c in cols + pulse_cols + extra_cols if c in df.columns]
    print(task_title)
    display(df[show].sort_values(['case', 'engine']).reset_index(drop=True))
    comparable_cases = sorted(df.loc[df['compare_status'] == 'pointwise-comparable', 'case'].unique())
    print('Pointwise-comparable cases =', comparable_cases)
    for metric in metrics:
        if metric in df.columns:
            plot_metric(df, metric, f'{task_title}: {metric}')


def collect_all(*frames: pd.DataFrame) -> pd.DataFrame:
    df = pd.concat(frames, ignore_index=True)
    df = attach_compare_status(df)
    out_csv = SUMMARY_ROOT / 'all_tasks_summary.csv'
    df.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print('all summary ->', out_csv)
    return df


## 1. 单比特基准模型与参数设置

目标：给出单比特基准场景，并比较不同静态参数设置下三引擎的终态表示与基础观测量。


In [4]:
task1_cases = [
    {
        'tag': 'baseline',
        'hardware': {'qubit_freqs_hz': [5.00], 'gate_duration': 20.0},
        'noise': {'model': 'markovian_lindblad', 't1': 120.0, 't2': 90.0},
        'note': 'baseline single-qubit model',
    },
    {
        'tag': 'detuned',
        'hardware': {'qubit_freqs_hz': [5.20], 'gate_duration': 28.0},
        'noise': {'model': 'markovian_lindblad', 't1': 80.0, 't2': 55.0},
        'note': 'detuned and slower gate',
    },
]

df_task1 = run_tri_cases('task1_single_qubit_baseline', QASM_1Q_BASE, task1_cases)
display_task(df_task1, 'Task 1 单比特基准模型', ['final_p1_obs', 'final_state_last'])


[task1_single_qubit_baseline] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task1_single_qubit_baseline_summary.csv
Task 1 单比特基准模型


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task1_single_qubit_baseline,baseline,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.3421061546928099, 0.6578938453071901]",1.000000,0.342106,0.657894,0.665048,260.0,158.4,0.8,16.400267,quantumoptics.timeevolution.master,True,baseline single-qubit model
1,task1_single_qubit_baseline,baseline,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.3421085300810429, 0.6578914699189571]",1.000000,0.342109,0.657891,0.626677,260.0,158.4,0.8,16.400267,quantumtoolbox.mesolve,True,baseline single-qubit model
2,task1_single_qubit_baseline,baseline,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.9997596199783175],0.999760,0.000240,0.999760,0.879648,260.0,158.4,0.8,16.400267,,False,baseline single-qubit model
3,task1_single_qubit_baseline,detuned,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.35332307589571843, 0.6466769241042816]",1.000000,0.353323,0.646677,0.651699,284.0,158.4,0.8,22.962654,quantumoptics.timeevolution.master,True,detuned and slower gate
4,task1_single_qubit_baseline,detuned,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.35333127697369227, 0.6466687230263077]",1.000000,0.353331,0.646669,0.627742,284.0,158.4,0.8,22.962654,quantumtoolbox.mesolve,True,detuned and slower gate
5,task1_single_qubit_baseline,detuned,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.9999962818466597],0.999996,0.000004,0.999996,0.919580,284.0,158.4,0.8,22.962654,,False,detuned and slower gate


Pointwise-comparable cases = []
Task 1 单比特基准模型: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 1 单比特基准模型: final_state_last: skipped plot because no case is pointwise-comparable across engines.


## 2. 读取线的输入与输出

说明：当前仓库没有完整的读取链路物理模型，因此这里用 `measure_duration` 与 `OU` 噪声做代理，并直接从 `pulse_samples.npz` 中提取 `RO_0` 通道的持续时间、面积和峰值，作为读取线 I/O 指标。


In [5]:
task2_cases = [
    {
        'tag': 'short_ro',
        'hardware': {'measure_duration': 40.0, 'gate_duration': 16.0},
        'noise': {'model': 'ou', 'ou_sigma': 0.01, 'ou_tau': 6.0},
        'note': 'short readout pulse',
    },
    {
        'tag': 'long_ro',
        'hardware': {'measure_duration': 120.0, 'gate_duration': 16.0},
        'noise': {'model': 'ou', 'ou_sigma': 0.03, 'ou_tau': 10.0},
        'note': 'longer readout pulse and stronger colored noise',
    },
]

df_task2 = run_tri_cases('task2_readout_io', QASM_1Q_READOUT, task2_cases)
display_task(df_task2, 'Task 2 读取线输入与输出', ['RO_0_duration', 'RO_0_abs_area', 'final_p1_obs'])


[task2_readout_io] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task2_readout_io_summary.csv
Task 2 读取线输入与输出


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task2_readout_io,long_ro,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.0,0.0,1.0,1.0,136.0,94.4,0.8,6.559366,quantumoptics.timeevolution.master,True,longer readout pulse and stronger colored noise
1,task2_readout_io,long_ro,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[1.0, 0.0]",1.0,1.0,0.0,0.0,136.0,94.4,0.8,6.559366,quantumtoolbox.sesolve,True,longer readout pulse and stronger colored noise
2,task2_readout_io,long_ro,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.0],0.0,1.0,0.0,0.0,136.0,94.4,0.8,6.559366,,False,longer readout pulse and stronger colored noise
3,task2_readout_io,short_ro,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.0,0.0,1.0,1.0,56.0,30.4,0.8,6.559366,quantumoptics.timeevolution.master,True,short readout pulse
4,task2_readout_io,short_ro,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[1.0, 0.0]",1.0,1.0,0.0,0.0,56.0,30.4,0.8,6.559366,quantumtoolbox.sesolve,True,short readout pulse
5,task2_readout_io,short_ro,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.0],0.0,1.0,0.0,0.0,56.0,30.4,0.8,6.559366,,False,short readout pulse


Pointwise-comparable cases = []
Task 2 读取线输入与输出: RO_0_duration: skipped plot because no case is pointwise-comparable across engines.
Task 2 读取线输入与输出: RO_0_abs_area: skipped plot because no case is pointwise-comparable across engines.
Task 2 读取线输入与输出: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.


## 3. 弛豫与退相干噪声的动力学影响

目标：对比较弱与较强 `T1 / T2` 条件下的动力学结果。


In [6]:
task3_cases = [
    {
        'tag': 'weak_relaxation',
        'hardware': {'gate_duration': 20.0},
        'noise': {'model': 'markovian_lindblad', 't1': 140.0, 't2': 100.0},
        'note': 'long coherence times',
    },
    {
        'tag': 'strong_relaxation',
        'hardware': {'gate_duration': 32.0},
        'noise': {'model': 'markovian_lindblad', 't1': 25.0, 't2': 18.0},
        'note': 'short coherence times',
    },
]

df_task3 = run_tri_cases('task3_relaxation_dephasing', QASM_1Q_READOUT, task3_cases)
display_task(df_task3, 'Task 3 弛豫与退相干', ['final_p1_obs', 'mean_excited_obs'])


[task3_relaxation_dephasing] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task3_relaxation_dephasing_summary.csv
Task 3 弛豫与退相干


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task3_relaxation_dephasing,strong_relaxation,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.000000,0.000000e+00,1.000000,1.000000,232.0,158.4,0.8,13.121836,quantumoptics.timeevolution.master,True,short coherence times
1,task3_relaxation_dephasing,strong_relaxation,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[5.473884012729968e-09, 0.999999994526116]",1.000000,5.473884e-09,1.000000,0.974522,232.0,158.4,0.8,13.121836,quantumtoolbox.mesolve,True,short coherence times
2,task3_relaxation_dephasing,strong_relaxation,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[1.0],1.000000,0.000000e+00,1.000000,0.974522,232.0,158.4,0.8,13.121836,,False,short coherence times
3,task3_relaxation_dephasing,weak_relaxation,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.000000,0.000000e+00,1.000000,1.000000,220.0,158.4,0.8,8.200134,quantumoptics.timeevolution.master,True,long coherence times
4,task3_relaxation_dephasing,weak_relaxation,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.000790491016174566, 0.9992095089838254]",1.000000,7.904910e-04,0.999210,0.859750,220.0,158.4,0.8,8.200134,quantumtoolbox.mesolve,True,long coherence times
5,task3_relaxation_dephasing,weak_relaxation,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.9992094878117312],0.999209,7.905122e-04,0.999209,0.859750,220.0,158.4,0.8,8.200134,,False,long coherence times


Pointwise-comparable cases = []
Task 3 弛豫与退相干: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 3 弛豫与退相干: mean_excited_obs: skipped plot because no case is pointwise-comparable across engines.


## 4. 非静态噪声（频漂、1/f）的数值特征

这里分别选一个 `1/f` 代理场景和一个 `OU` 频漂场景，比较三引擎输出。


In [7]:
task4_cases = [
    {
        'tag': 'one_over_f',
        'hardware': {'gate_duration': 22.0},
        'noise': {'model': 'one_over_f', 'one_over_f_amp': 0.04, 'one_over_f_fmin': 1e-3, 'one_over_f_fmax': 0.10},
        'note': '1/f surrogate noise',
    },
    {
        'tag': 'ou_drift',
        'hardware': {'gate_duration': 22.0},
        'noise': {'model': 'ou', 'ou_sigma': 0.04, 'ou_tau': 12.0},
        'note': 'OU frequency drift surrogate',
    },
]

df_task4 = run_tri_cases('task4_nonstationary_noise', QASM_1Q_BASE, task4_cases)
display_task(df_task4, 'Task 4 非静态噪声', ['final_p1_obs', 'final_state_last'])


[task4_nonstationary_noise] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task4_nonstationary_noise_summary.csv
Task 4 非静态噪声


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task4_nonstationary_noise,one_over_f,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.0,0.0,1.0,1.0,266.0,158.4,0.8,18.040927,quantumoptics.timeevolution.master,True,1/f surrogate noise
1,task4_nonstationary_noise,one_over_f,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[1.0, 0.0]",1.0,1.0,0.0,0.0,266.0,158.4,0.8,18.040927,quantumtoolbox.sesolve,True,1/f surrogate noise
2,task4_nonstationary_noise,one_over_f,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.0],0.0,1.0,0.0,0.0,266.0,158.4,0.8,18.040927,,False,1/f surrogate noise
3,task4_nonstationary_noise,ou_drift,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.0,0.0,1.0,1.0,266.0,158.4,0.8,18.040927,quantumoptics.timeevolution.master,True,OU frequency drift surrogate
4,task4_nonstationary_noise,ou_drift,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[1.0, 0.0]",1.0,1.0,0.0,0.0,266.0,158.4,0.8,18.040927,quantumtoolbox.sesolve,True,OU frequency drift surrogate
5,task4_nonstationary_noise,ou_drift,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.0],0.0,1.0,0.0,0.0,266.0,158.4,0.8,18.040927,,False,OU frequency drift surrogate


Pointwise-comparable cases = []
Task 4 非静态噪声: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 4 非静态噪声: final_state_last: skipped plot because no case is pointwise-comparable across engines.


## 5. 控制与读取链路噪声的联合影响

这里把 `control_scale` 与读取链路代理噪声组合起来，观察联合扰动下的结果差异。


In [8]:
task5_cases = [
    {
        'tag': 'nominal_joint',
        'hardware': {'control_scale': 0.90, 'measure_duration': 60.0, 'gate_duration': 20.0},
        'noise': {'model': 'ou', 'ou_sigma': 0.01, 'ou_tau': 6.0},
        'note': 'mild control/readout disturbance',
    },
    {
        'tag': 'stressed_joint',
        'hardware': {'control_scale': 1.25, 'measure_duration': 120.0, 'gate_duration': 30.0},
        'noise': {'model': 'ou', 'ou_sigma': 0.05, 'ou_tau': 14.0},
        'note': 'stronger combined control/readout disturbance',
    },
]

df_task5 = run_tri_cases('task5_joint_control_readout', QASM_1Q_BASE, task5_cases)
display_task(df_task5, 'Task 5 控制与读取链路联合影响', ['RO_0_abs_area', 'final_p1_obs', 'final_state_last'])


[task5_joint_control_readout] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task5_joint_control_readout_summary.csv
Task 5 控制与读取链路联合影响


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task5_joint_control_readout,nominal_joint,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.0,0.0,1.0,1.0,120.0,46.4,0.8,16.400267,quantumoptics.timeevolution.master,True,mild control/readout disturbance
1,task5_joint_control_readout,nominal_joint,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[1.0, 0.0]",1.0,1.0,0.0,0.0,120.0,46.4,0.8,16.400267,quantumtoolbox.sesolve,True,mild control/readout disturbance
2,task5_joint_control_readout,nominal_joint,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.0],0.0,1.0,0.0,0.0,120.0,46.4,0.8,16.400267,,False,mild control/readout disturbance
3,task5_joint_control_readout,stressed_joint,julia_qoptics,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[0.0, 1.0]",1.0,0.0,1.0,1.0,210.0,94.4,0.8,24.603173,quantumoptics.timeevolution.master,True,stronger combined control/readout disturbance
4,task5_joint_control_readout,stressed_joint,julia_qtoolbox,basis_population_single_qubit,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,2,"[1.0, 0.0]",1.0,1.0,0.0,0.0,210.0,94.4,0.8,24.603173,quantumtoolbox.sesolve,True,stronger combined control/readout disturbance
5,task5_joint_control_readout,stressed_joint,qutip,per_qubit_excited_probability,semantic-review-needed,basis_population_single_qubit | per_qubit_exci...,1,[0.0],0.0,1.0,0.0,0.0,210.0,94.4,0.8,24.603173,,False,stronger combined control/readout disturbance


Pointwise-comparable cases = []
Task 5 控制与读取链路联合影响: RO_0_abs_area: skipped plot because no case is pointwise-comparable across engines.
Task 5 控制与读取链路联合影响: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 5 控制与读取链路联合影响: final_state_last: skipped plot because no case is pointwise-comparable across engines.


## 6. 两比特耦合与两比特门操作

使用 Bell 型电路，比较较弱/较强耦合参数下三引擎输出。


In [9]:
task6_cases = [
    {
        'tag': 'weak_coupling',
        'hardware': {'couplings': [{'i': 0, 'j': 1, 'g': 0.02, 'kind': 'xx+yy'}], 'gate_duration': 20.0},
        'noise': {'model': 'markovian_lindblad', 't1': [100.0, 100.0], 't2': [80.0, 80.0]},
        'note': 'weak 2Q coupling',
    },
    {
        'tag': 'strong_coupling',
        'hardware': {'couplings': [{'i': 0, 'j': 1, 'g': 0.08, 'kind': 'xx+yy'}], 'gate_duration': 28.0},
        'noise': {'model': 'markovian_lindblad', 't1': [60.0, 65.0], 't2': [45.0, 50.0]},
        'note': 'stronger 2Q coupling',
    },
]

df_task6 = run_tri_cases('task6_two_qubit_coupling', QASM_2Q_BELL, task6_cases)
display_task(df_task6, 'Task 6 两比特耦合与两比特门', ['final_p1_obs', 'final_state_last'])


[task6_two_qubit_coupling] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task6_two_qubit_coupling_summary.csv
Task 6 两比特耦合与两比特门


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task6_two_qubit_coupling,strong_coupling,julia_qoptics,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.0, 1.0]",1.000000,NaN,NaN,NaN,284.0,158.4,0.8,32.520308,quantumoptics.timeevolution.master,True,stronger 2Q coupling
1,task6_two_qubit_coupling,strong_coupling,julia_qtoolbox,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[5.124667357137014e-11, 0.9999999999487533]",1.000000,NaN,NaN,NaN,284.0,158.4,0.8,32.520308,quantumtoolbox.mesolve,True,stronger 2Q coupling
2,task6_two_qubit_coupling,strong_coupling,qutip,per_qubit_excited_probability,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.999999890485117, 0.999999890095249]",2.000000,1.095149e-07,1.000000,0.937160,284.0,158.4,0.8,32.520308,,False,stronger 2Q coupling
3,task6_two_qubit_coupling,weak_coupling,julia_qoptics,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.0, 1.0]",1.000000,NaN,NaN,NaN,260.0,158.4,0.8,23.228062,quantumoptics.timeevolution.master,True,weak 2Q coupling
4,task6_two_qubit_coupling,weak_coupling,julia_qtoolbox,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[2.1175203812617838e-09, 0.9999999978824796]",1.000000,NaN,NaN,NaN,260.0,158.4,0.8,23.228062,quantumtoolbox.mesolve,True,weak 2Q coupling
5,task6_two_qubit_coupling,weak_coupling,qutip,per_qubit_excited_probability,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.9999545932390769, 0.9999545932390769]",1.999909,4.540676e-05,0.999955,0.899604,260.0,158.4,0.8,23.228062,,False,weak 2Q coupling


Pointwise-comparable cases = []
Task 6 两比特耦合与两比特门: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 6 两比特耦合与两比特门: final_state_last: skipped plot because no case is pointwise-comparable across engines.


## 7. 量子比特、控制线与读取线的串扰

说明：当前没有显式控制线/读取线串扰网络模型，这里通过“驱动 `q0` + spectator `q1` + 非零耦合 + 彩色噪声”来构造串扰代理场景。


In [10]:
task7_cases = [
    {
        'tag': 'weak_xtalk',
        'hardware': {
            'couplings': [{'i': 0, 'j': 1, 'g': 0.02, 'kind': 'xx+yy'}],
            'qubit_freqs_hz': [5.00, 5.15],
            'measure_duration': 60.0,
            'gate_duration': 20.0,
        },
        'noise': {'model': 'ou', 'ou_sigma': [0.01, 0.01], 'ou_tau': [8.0, 8.0]},
        'note': 'weak crosstalk proxy',
    },
    {
        'tag': 'strong_xtalk',
        'hardware': {
            'couplings': [{'i': 0, 'j': 1, 'g': 0.10, 'kind': 'xx+yy'}],
            'qubit_freqs_hz': [5.00, 5.05],
            'measure_duration': 120.0,
            'gate_duration': 28.0,
        },
        'noise': {'model': 'ou', 'ou_sigma': [0.04, 0.04], 'ou_tau': [12.0, 12.0]},
        'note': 'strong crosstalk proxy',
    },
]

df_task7 = run_tri_cases('task7_crosstalk_proxy', QASM_2Q_SPECTATOR, task7_cases)
display_task(df_task7, 'Task 7 串扰代理场景', ['RO_0_abs_area', 'final_p1_obs', 'final_state_last'])


[task7_crosstalk_proxy] wrote D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\task7_crosstalk_proxy_summary.csv
Task 7 串扰代理场景


,task,case,engine,state_encoding,compare_status,compare_reason,state_len,final_state_json,final_state_sum,final_p0_obs,final_p1_obs,mean_excited_obs,RO_0_duration,RO_0_abs_area,RO_0_peak,XY_0_abs_area,solver_impl,native_solver,note
0,task7_crosstalk_proxy,strong_xtalk,julia_qoptics,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.4477310030809296, 0.5522689969190704]",1.0,NaN,NaN,NaN,176.0,94.4,0.8,11.481327,quantumoptics.timeevolution.master,True,strong crosstalk proxy
1,task7_crosstalk_proxy,strong_xtalk,julia_qtoolbox,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.5521014011214156, 0.44789859887858435]",1.0,NaN,NaN,NaN,176.0,94.4,0.8,11.481327,quantumtoolbox.sesolve,True,strong crosstalk proxy
2,task7_crosstalk_proxy,strong_xtalk,qutip,per_qubit_excited_probability,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.0, 0.0]",0.0,1.0,0.0,0.0,176.0,94.4,0.8,11.481327,,False,strong crosstalk proxy
3,task7_crosstalk_proxy,weak_xtalk,julia_qoptics,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.4477310030809296, 0.5522689969190704]",1.0,NaN,NaN,NaN,100.0,46.4,0.8,8.200134,quantumoptics.timeevolution.master,True,weak crosstalk proxy
4,task7_crosstalk_proxy,weak_xtalk,julia_qtoolbox,ambiguous_population_vector,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.5521014011214156, 0.44789859887858435]",1.0,NaN,NaN,NaN,100.0,46.4,0.8,8.200134,quantumtoolbox.sesolve,True,weak crosstalk proxy
5,task7_crosstalk_proxy,weak_xtalk,qutip,per_qubit_excited_probability,semantic-review-needed,ambiguous_population_vector | per_qubit_excite...,2,"[0.0, 0.0]",0.0,1.0,0.0,0.0,100.0,46.4,0.8,8.200134,,False,weak crosstalk proxy


Pointwise-comparable cases = []
Task 7 串扰代理场景: RO_0_abs_area: skipped plot because no case is pointwise-comparable across engines.
Task 7 串扰代理场景: final_p1_obs: skipped plot because no case is pointwise-comparable across engines.
Task 7 串扰代理场景: final_state_last: skipped plot because no case is pointwise-comparable across engines.


## 汇总

把 7 个任务的摘要表合并，并写到 `examples/noise_simulation_tests/runs/required_tasks_tri_engine/_summaries/all_tasks_summary.csv`。


In [11]:
all_df = collect_all(df_task1, df_task2, df_task3, df_task4, df_task5, df_task6, df_task7)
summary = (
    all_df.groupby(['task', 'engine', 'state_encoding', 'compare_status'])[['state_len', 'final_state_sum', 'final_p1_obs', 'mean_excited_obs']]
    .agg(['mean', 'min', 'max'])
)
display(summary)
all_df.head()


all summary -> D:\超导量子计算机噪声抑制\qsim\examples\noise_simulation_tests\runs\required_tasks_tri_engine\_summaries\all_tasks_summary.csv


state_len  \
                                                                                                     mean   
task                        engine         state_encoding                compare_status                     
task1_single_qubit_baseline julia_qoptics  basis_population_single_qubit semantic-review-needed       2.0   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       1.0   
task2_readout_io            julia_qoptics  basis_population_single_qubit semantic-review-needed       2.0   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       1.0   
task3_relaxation_dephasing  julia_qoptics  basis_population_single_qubit semantic-review-needed       2.0   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       1.0   
task4_nonstationary_noise   julia_qoptics  basis_population_single_qubit semantic-review-needed       2.0   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       1.0   
task5_joint_control_readout julia_qoptics  basis_population_single_qubit semantic-review-needed       2.0   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       1.0   
task6_two_qubit_coupling    julia_qoptics  ambiguous_population_vector   semantic-review-needed       2.0   
                            julia_qtoolbox ambiguous_population_vector   semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       2.0   
task7_crosstalk_proxy       julia_qoptics  ambiguous_population_vector   semantic-review-needed       2.0   
                            julia_qtoolbox ambiguous_population_vector   semantic-review-needed       2.0   
                            qutip          per_qubit_excited_probability semantic-review-needed       2.0   

                                                                                                     \
                                                                                                min   
task                        engine         state_encoding                compare_status               
task1_single_qubit_baseline julia_qoptics  basis_population_single_qubit semantic-review-needed   2   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed   2   
                            qutip          per_qubit_excited_probability semantic-review-needed   1   
task2_readout_io            julia_qoptics  basis_population_single_qubit semantic-review-needed   2   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed   2   
                            qutip          per_qubit_excited_probability semantic-review-needed   1   
task3_relaxation_dephasing  julia_qoptics  basis_population_single_qubit semantic-review-needed   2   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed   2   
                            qutip          per_qubit_excited_probability semantic-review-needed   1   
task4_nonstationary_noise   julia_qoptics  basis_population_single_qubit semantic-review-needed   2   
                            julia_qtoolbox basis_population_single_qubit semantic-review-needed   2   
                            qutip    

,task,case,engine,trace_engine,state_encoding,num_qubits,state_len,final_state_json,final_state_sum,final_state_last,...,RO_1_abs_area,RO_1_peak,TC_0_samples,TC_0_duration,TC_0_abs_area,TC_0_peak,XY_1_samples,XY_1_duration,XY_1_abs_area,XY_1_peak
0,task1_single_qubit_baseline,baseline,qutip,qutip,per_qubit_excited_probability,1,1,[0.9997596199783175],0.999760,0.999760,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,task1_single_qubit_baseline,baseline,julia_qtoolbox,julia-quantumtoolbox,basis_population_single_qubit,1,2,"[0.3421085300810429, 0.6578914699189571]",1.000000,0.657891,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,task1_single_qubit_baseline,baseline,julia_qoptics,julia-quantumoptics,basis_population_single_qubit,1,2,"[0.3421061546928099, 0.6578938453071901]",1.000000,0.657894,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,task1_single_qubit_baseline,detuned,qutip,qutip,per_qubit_excited_probability,1,1,[0.9999962818466597],0.999996,0.999996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,task1_single_qubit_baseline,detuned,julia_qtoolbox,julia-quantumtoolbox,basis_population_single_qubit,1,2,"[0.35333127697369227, 0.6466687230263077]",1.000000,0.646669,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## ?????

- ? notebook ?? 7 ???????????????????? `examples/noise_simulation_tests/runs/required_tasks_tri_engine/`?
- ??????????? `state_encoding` ? `compare_status`??? `per_qubit_excited_probability` ??????????????
- `Task 2 / Task 5 / Task 7` ??????????????????/?????
- ?????????????? Julia ?????? `ambiguous_population_vector`????????????????????????
